In [1]:
!pip install -q vllm
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 98.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
import vllm
print("✅ vLLM imported successfully!")
print(f"vLLM version: {vllm.__version__}")

✅ vLLM imported successfully!
vLLM version: 0.29.0


In [3]:
!pip install --upgrade huggingface-hub
!pip install --upgrade vllm

In [4]:
pip install -U "torchao>=0.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
!pip install math_verify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 5.2 MB/s eta 0:00:00:00:01


In [ ]:
# -*- coding: utf-8 -*-
"""
Complete SVAMP evaluation with fixed vLLM (no BOS duplication).
Evaluates:
- base
- DistilledRL (V8_fixed) steps 80, 100, 120, 140, 160
- GRPO step 80

Saves and uploads completions & summaries to HF.
"""

import os
import json
import re
import math
import torch
from tqdm import tqdm
from datasets import load_dataset
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import HfApi, hf_hub_download, login

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# Define models to evaluate
# Format: (name, adapter_repo, step, is_adapter)
MODELS = [
    ("base", None, None, False),
    ("Gated_V1_step40", "HFrepo/Gated_DistilledRL_V1", 40, True),
    ("Gated_V1_step60", "HFrepo/Gated_DistilledRL_V1", 60, True),
    ("Gated_V1_step80", "HFrepo/Gated_DistilledRL_V1", 80, True),
    ("v12_step40", "HFrepo/DistilledRL_V12", 40, True),
    ("v12_step60", "HFrepo/DistilledRL_V12", 60, True),
    ("grpoV12_step40", "HFrepo/GRPO_V12", 40, True),
    ("grpoV12_KL_step40", "HFrepo/GRPO_V12_KL", 40, True),
]



# Evaluation settings (exactly matching original)
TEMPERATURE = 0.1
TOP_P = 0.95
TOP_K = 0
MAX_NEW_TOKENS = 512
PROMPT_MAX_LEN = 512
N_SAMPLES = 32
N_QUESTIONS = 300

# Output and HF
RESULTS_REPO = "HFrepo/DistilledRL_Eval_Results"
OUTPUT_DIR = "./svamp_fixed_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# vLLM settings
GPU_MEMORY_UTIL = 0.75
BATCH_SIZE = 50
SAVE_EVERY = 10
SEED = 42

K_VALUES = [1, 2, 4, 8, 16, 32]

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        HF_TOKEN = secrets.get_secret("HF_TOKEN")
    except:
        pass
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found.")
login(token=HF_TOKEN)
api = HfApi()

# ============================================================
# LOAD SVAMP (first 300)
# ============================================================

print("Loading SVAMP (first 300)...")
try:
    ds = load_dataset("ChilleD/SVAMP", split="test", token=HF_TOKEN)
except:
    ds = load_dataset("google-research-datasets/svamp", split="test", token=HF_TOKEN)
ds = ds.select(range(min(N_QUESTIONS, len(ds))))
n_total = len(ds)
print(f"Loaded {n_total} questions.")

# We'll build prompt strings once – they are the same for all models
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

def build_prompt(body, question):
    user_content = (
        "Please reason step by step, and put your final answer within \\boxed{}.\n\n"
        f"{body}\n\n"
        f"Question: {question}"
    )
    chat = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

prompt_strings = [build_prompt(ex["Body"], ex["Question"]) for ex in ds]
gts = [str(ex["Answer"]).strip() for ex in ds]
questions = [ex["Question"] for ex in ds]
bodies = [ex["Body"] for ex in ds]

# ============================================================
# JUDGE FUNCTION (identical to original)
# ============================================================

def parse_number(s):
    if s is None: return None
    s = str(s).strip().replace(",", "").replace("$", "").replace("%", "")
    s = s.rstrip(".:;, ")
    frac = re.search(r"(-?\d+(?:\.\d+)?)\s*/\s*(-?\d+(?:\.\d+)?)", s)
    if frac:
        num, den = float(frac.group(1)), float(frac.group(2))
        return num/den if den != 0 else None
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else None

def extract_boxed_content_fallback(text):
    indices = [m.start() for m in re.finditer(r"\\boxed", text)]
    if not indices: return None
    idx = indices[-1]
    i = text.find("{", idx)
    if i == -1: return None
    depth = 0
    for j in range(i, len(text)):
        if text[j] == "{": depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i+1:j]
    return None

ANSWER_PATTERNS = [
    r"(?:final\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
    r"(?:the\s+answer)\s*(?:is|=|:)\s*([^\n.]+)",
    r"(?:therefore|thus|so)\s*,?\s*(?:the\s+)?answer\s*(?:is|=|:)\s*([^\n.]+)",
]

def extract_answer_regex(text):
    if not text: return None
    boxed = extract_boxed_content_fallback(text)
    if boxed is not None:
        val = parse_number(boxed)
        if val is not None: return val
    for pat in ANSWER_PATTERNS:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            val = parse_number(matches[-1])
            if val is not None: return val
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if sentences:
        val = parse_number(sentences[-1])
        if val is not None: return val
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if nums:
        val = parse_number(nums[-1])
        if val is not None: return val
    return None

def judge_math_answer(completion, gold_answer):
    try:
        from math_verify import parse, verify
        from math_verify.parser import LatexExtractionConfig, ExprExtractionConfig
        pred_parsed = parse(completion, extraction_config=[ExprExtractionConfig(), LatexExtractionConfig()])
        gold_text = str(gold_answer).strip()
        if "\\boxed" not in gold_text and "$" not in gold_text:
            gold_text = f"\\boxed{{{gold_text}}}"
        gold_parsed = parse(gold_text, extraction_config=[LatexExtractionConfig(), ExprExtractionConfig()])
        if gold_parsed and pred_parsed:
            return bool(verify(gold_parsed, pred_parsed))
    except:
        pass
    pred_num = extract_answer_regex(completion)
    gt_num = parse_number(gold_answer)
    if pred_num is not None and gt_num is not None:
        return abs(pred_num - gt_num) < 1e-4
    return False

def judge_records(records):
    binary_pass1 = []
    pass_at_k = {k: [] for k in K_VALUES}
    total_correct = 0
    total_samples = 0
    for r in records:
        completions = r.get("completions", [])
        gt = r.get("gt", "")
        while len(completions) < N_SAMPLES:
            completions.append(None)
        flags = [judge_math_answer(c, gt) if c is not None else False for c in completions]
        c = sum(flags)
        n = len(flags)
        total_correct += c
        total_samples += n
        binary_pass1.append(1.0 if flags[0] else 0.0)
        for k in K_VALUES:
            val = 1.0 - math.prod((n - c - i) / (n - i) for i in range(k)) if n - c >= k else 1.0
            pass_at_k[k].append(val)
    summary = {
        "n_questions": len(records),
        "empirical_accuracy": total_correct / total_samples if total_samples else 0.0,
        "pass_at_k": {f"pass@{k}": sum(pass_at_k[k]) / len(pass_at_k[k]) for k in K_VALUES},
    }
    return binary_pass1, summary

# ============================================================
# UTILITY: LOAD/SAVE RECORDS
# ============================================================

def get_existing_records(name):
    local_file = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    if os.path.exists(local_file):
        with open(local_file, "r") as f:
            records = [json.loads(line) for line in f]
        print(f"[{name}] Loaded {len(records)} from local.")
        return records
    # Try to download from HF
    try:
        hf_hub_download(
            repo_id=RESULTS_REPO,
            filename=f"{name}.jsonl",
            local_dir=OUTPUT_DIR,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        if os.path.exists(local_file):
            with open(local_file, "r") as f:
                records = [json.loads(line) for line in f]
            print(f"[{name}] Loaded {len(records)} from HF.")
            return records
    except:
        pass
    print(f"[{name}] No existing cache.")
    return []

def save_records(name, records, upload=True):
    local_file = os.path.join(OUTPUT_DIR, f"{name}.jsonl")
    with open(local_file, "w") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    if upload:
        api.create_repo(RESULTS_REPO, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=local_file,
            path_in_repo=f"{name}.jsonl",
            repo_id=RESULTS_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
            commit_message=f"SVAMP fixed vLLM {name}",
        )
        print(f"[{name}] Uploaded {len(records)} records.")

# ============================================================
# MERGE ADAPTER (if needed)
# ============================================================

def merge_adapter(repo, step):
    merged_path = f"./merged_adapters/{repo.split('/')[-1]}_step{step}"
    if os.path.exists(os.path.join(merged_path, "config.json")):
        print(f"Merged model already exists at {merged_path}.")
        return merged_path

    print(f"Merging adapter from {repo} (step {step})...")
    local_adapter_dir = f"./adapter_cache/{repo.split('/')[-1]}/step{step}"
    os.makedirs(local_adapter_dir, exist_ok=True)
    try:
        hf_hub_download(
            repo_id=repo,
            filename=f"step{step}/adapter_config.json",
            local_dir=f"./adapter_cache/{repo.split('/')[-1]}",
            repo_type="model",
            token=HF_TOKEN,
        )
        hf_hub_download(
            repo_id=repo,
            filename=f"step{step}/adapter_model.safetensors",
            local_dir=f"./adapter_cache/{repo.split('/')[-1]}",
            repo_type="model",
            token=HF_TOKEN,
        )
    except:
        hf_hub_download(
            repo_id=repo,
            filename=f"step{step}/adapter_model.bin",
            local_dir=f"./adapter_cache/{repo.split('/')[-1]}",
            repo_type="model",
            token=HF_TOKEN,
        )

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, local_adapter_dir)
    merged = model.merge_and_unload()
    os.makedirs(merged_path, exist_ok=True)
    merged.save_pretrained(merged_path)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tokenizer.save_pretrained(merged_path)
    print(f"Merged model saved to {merged_path}")
    return merged_path

# ============================================================
# GENERATE COMPLETIONS FOR A MODEL (fixed vLLM)
# ============================================================

def generate_for_model(name, model_path, is_adapter=False, repo=None, step=None):
    # First check if we already have records
    existing_records = get_existing_records(name)
    if len(existing_records) == n_total:
        print(f"[{name}] Already complete. Skipping generation.")
        return existing_records

    # If not complete, we need to generate missing questions
    existing_ids = {r.get("question_index", -1) for r in existing_records}
    all_ids = set(range(n_total))
    missing_ids = all_ids - existing_ids
    print(f"[{name}] Need to generate {len(missing_ids)} missing questions.")

    # Merge if adapter
    if is_adapter:
        model_path = merge_adapter(repo, step)

    # Load tokenizer from model path
    vllm_tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    vllm_tokenizer.padding_side = "left"

    # Tokenize prompts manually (no extra BOS)
    prompt_token_ids = [
        vllm_tokenizer.encode(p, add_special_tokens=False) for p in prompt_strings
    ]

    # Initialize vLLM
    llm = LLM(
        model=model_path,
        dtype="float16",
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        max_model_len=PROMPT_MAX_LEN + MAX_NEW_TOKENS + 64,
        trust_remote_code=True,
        seed=SEED,
    )

    sampling_params = SamplingParams(
        n=N_SAMPLES,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        max_tokens=MAX_NEW_TOKENS,
        repetition_penalty=1.0,
        presence_penalty=0.0,
        frequency_penalty=0.0,
        seed=SEED,
    )

    # Generate only missing questions
    missing_list = sorted(missing_ids)
    new_records = []
    for i in tqdm(range(0, len(missing_list), BATCH_SIZE), desc=f"{name} batches"):
        batch_ids = missing_list[i:i+BATCH_SIZE]
        batch_token_ids = [prompt_token_ids[idx] for idx in batch_ids]
        outputs = llm.generate(batch_token_ids, sampling_params)
        for q_idx, out in zip(batch_ids, outputs):
            completions = [o.text for o in out.outputs]
            new_records.append({
                "question_index": q_idx,
                "body": bodies[q_idx],
                "question": questions[q_idx],
                "gt": gts[q_idx],
                "completions": completions,
            })
        # Save incrementally
        all_records = existing_records + new_records
        all_records = sorted(all_records, key=lambda x: x["question_index"])
        if len(new_records) >= SAVE_EVERY or (i + len(batch_ids)) >= len(missing_list):
            save_records(name, all_records, upload=True)

    all_records = existing_records + new_records
    all_records = sorted(all_records, key=lambda x: x["question_index"])
    save_records(name, all_records, upload=True)

    # Cleanup
    del llm
    torch.cuda.empty_cache()
    return all_records

# ============================================================
# EVALUATE ALL MODELS
# ============================================================

summaries = {}
binary_results = {}

for name, repo, step, is_adapter in MODELS:
    if is_adapter:
        model_path = None  # will be merged inside generate_for_model
    else:
        model_path = BASE_MODEL_ID

    print(f"\n{'='*80}")
    print(f"Processing {name}")
    print(f"{'='*80}")

    # Generate or load completions
    records = generate_for_model(name, model_path, is_adapter, repo, step)
    # Judge
    bin, summ = judge_records(records)
    summaries[name] = summ
    binary_results[name] = bin

    # Save summary locally and upload
    summary_file = os.path.join(OUTPUT_DIR, f"{name}_summary.json")
    with open(summary_file, "w") as f:
        json.dump(summ, f, indent=2)
    api.upload_file(
        path_or_fileobj=summary_file,
        path_in_repo=f"{name}_summary.json",
        repo_id=RESULTS_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message=f"SVAMP fixed vLLM summary {name}",
    )
    print(f"[{name}] Summary uploaded.")

# ============================================================
# COMPARISON TABLE
# ============================================================

print("\n" + "="*100)
print("SVAMP PASS@K COMPARISON (fixed vLLM)")
print("="*100)
header = f"{'Model':<20} | " + " | ".join(f"k={k:<5}" for k in K_VALUES) + " | Emp. Acc"
print(header)
print("-"*120)
for name in summaries:
    summ = summaries[name]
    row = f"{name:<20} | "
    values = [f"{summ['pass_at_k'][f'pass@{k}']*100:5.1f}%" for k in K_VALUES]
    row += " | ".join(values)
    row += f" | {summ['empirical_accuracy']*100:5.1f}%"
    print(row)

# ============================================================
# MCNEMAR TESTS (against base)
# ============================================================

def mcnemar(a, b):
    a_only = sum(1 for x, y in zip(a, b) if x and not y)
    b_only = sum(1 for x, y in zip(a, b) if y and not x)
    n = a_only + b_only
    if n == 0:
        return 1.0, "no disagreements"
    from math import comb, erfc, sqrt
    if n < 25:
        k = min(a_only, b_only)
        p = 2 * sum(comb(n, i) for i in range(k+1)) / (2 ** n)
        p = min(p, 1.0)
        method = "exact binomial"
    else:
        stat = (abs(a_only - b_only) - 1) ** 2 / n
        p = erfc(sqrt(stat / 2))
        method = "chi-square (cc)"
    return p, f"b={a_only}, c={b_only}, n={n} ({method})"

if "base" in binary_results:
    base_bin = binary_results["base"]
    print("\n" + "="*80)
    print("MCNEMAR TESTS (pass@1) vs base")
    print("="*80)
    for name in binary_results:
        if name == "base": continue
        p, details = mcnemar(base_bin, binary_results[name])
        print(f"\n{name} vs base:")
        print(f"  {details}")
        print(f"  p = {p:.6f}")
        print(f"  --> {'Significant' if p < 0.05 else 'Not significant'}")

# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

global_summary = {
    "eval_config": {
        "dataset": "SVAMP (first 300)",
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "max_new_tokens": MAX_NEW_TOKENS,
        "n_samples": N_SAMPLES,
        "generation_engine": "vLLM (fixed tokenization)",
        "seed": SEED,
    },
    "summaries": summaries,
}
global_file = os.path.join(OUTPUT_DIR, "svamp_fixed_vllm_global_summary.json")
with open(global_file, "w") as f:
    json.dump(global_summary, f, indent=2)
api.upload_file(
    path_or_fileobj=global_file,
    path_in_repo="svamp_fixed_vllm_global_summary.json",
    repo_id=RESULTS_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="SVAMP fixed vLLM global summary",
)
print("\nGlobal summary uploaded to HF.")
print("All done.")

W0920 08:59:08.802000 58 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0920 08:59:08.846000 58 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading SVAMP (first 300)...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  111kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

[grpoV12_step40] Summary uploaded.

Processing grpoV12_KL_step40


grpoV12_KL_step40.jsonl:   0%|          | 0.00/4.03M [00:00<?, ?B/s]

[grpoV12_KL_step40] Loaded 300 from HF.
[grpoV12_KL_step40] Already complete. Skipping generation.


No files have been modified since last commit. Skipping to prevent empty commit.


[grpoV12_KL_step40] Summary uploaded.

SVAMP PASS@K COMPARISON (fixed vLLM)
Model                | k=1     | k=2     | k=4     | k=8     | k=16    | k=32    | Emp. Acc
------------------------------------------------------------------------------------------------------------------------
base                 |  66.0% |  71.0% |  75.0% |  78.3% |  80.8% |  82.7% |  66.0%
Gated_V1_step40      |  61.8% |  68.4% |  73.5% |  77.8% |  81.7% |  84.7% |  61.8%
Gated_V1_step60      |  62.7% |  71.2% |  77.5% |  81.8% |  84.6% |  86.7% |  62.7%
Gated_V1_step80      |  63.2% |  70.8% |  76.6% |  80.8% |  83.7% |  86.0% |  63.2%
DRL_amplified_step40 |  62.5% |  69.1% |  74.6% |  79.1% |  82.7% |  85.7% |  62.5%
v12_step40           |  64.0% |  72.1% |  78.0% |  82.1% |  85.1% |  87.3% |  64.0%
v12_step60           |  64.1% |  74.0% |  80.9% |  86.1% |  89.9% |  92.3% |  64.1%
grpoV12_step40       |  40.2% |  44.5% |  48.2% |  51.3% |  53.8% |  55.3% |  40.2%
grpoV12_KL_step40    |  61.0% |  65.0% 